In [ ]:
import pandas as pd
from wsi_stats import WSIStatsCache

# Path to WSIs (root dir)
ROOT_DIR = "//regsj.intern/appl/Deep_Visual_Proteomics"

# Path to cache file
CACHE_FILE = "D:\DATA\wsi_cache.pkl"

# Load/reload WSI stats cache
wsi_cache = WSIStatsCache(ROOT_DIR, CACHE_FILE)
df_wsi = wsi_cache.main(reload=False)

In [ ]:
# Check for WSI with no associated data
missing_data = df_wsi[
    (df_wsi["file_size"].isna() | (df_wsi["file_size"] == 0)) |
    (df_wsi["data_folder_size"].isna() | (df_wsi["data_folder_size"] == 0))
]

# Save index of rows to drop
rows_to_drop = missing_data.index
print("WSIs with missing data:", len(missing_data))

In [ ]:
# Drop rows with no associated data
df_no_missing = df_wsi.drop(index=rows_to_drop)

print(f"Original rows: {len(df_wsi)}, After dropping: {len(df_no_missing)}")

In [ ]:
df_no_missing = df_no_missing.reset_index()  # filename becomes a column
print(df_no_missing.head())

In [ ]:
from ocr_labels import LabelOCRCache

all_filenames = df_no_missing["filename"].tolist()
print(f"Number of slides with data: {len(all_filenames)}")

CACHE_OCR = "D:\DATA\ocr_cache.pkl"

OCRcache = LabelOCRCache(all_filenames, CACHE_OCR)
df_ocr = OCRcache.run()
print(df_ocr.head())

In [ ]:
# Rename columns
df_ocr = df_ocr.rename(columns={
    "rekvnr": "ocr_rekvnr",
    "stain": "ocr_stain"
})

# Merge dataframes
df_merged = df_no_missing.merge(df_ocr, on="filename", how="left")
print(df_merged.head())

In [ ]:
# Path to pathology metadata
df_path = r"D:\DATA\full_dataset\initial_cleaning.csv"
df_pathology = pd.read_csv(df_path)
print(df_pathology.head())

In [ ]:
# Convert rekvnr to int (ensure same formatting)
df_pathology["rekvnr"] = pd.to_numeric(df_pathology["rekvnr"], errors="coerce").astype("Int64")
df_merged["rekvnr"] = pd.to_numeric(df_merged["rekvnr"], errors="coerce").astype("Int64")
df_merged["ocr_rekvnr"] = pd.to_numeric(df_merged["ocr_rekvnr"], errors="coerce").astype("Int64")

In [ ]:
# Set of all unique rekvnr in pathology df  
rekvnr_pathology = set(df_pathology["rekvnr"])

print(f"unique rekvnr in pathology df: {len(rekvnr_pathology)}")

In [ ]:
def determine_true_rekvnr(row):
    rekvnr = row["rekvnr"]
    ocr = row["ocr_rekvnr"]

    # Rule 1: filename and OCR agree, and IS in pathology → trust it
    if pd.notna(rekvnr) and pd.notna(ocr) and rekvnr == ocr and (rekvnr in rekvnr_pathology):
        return rekvnr

    # Rule 2: filename NOT in pathology, OCR IS in pathology → OCR is true
    if (rekvnr not in rekvnr_pathology) and (ocr in rekvnr_pathology):
        return ocr

    # Rule 3: filename IS in pathology → trust filename
    if rekvnr in rekvnr_pathology:
        return rekvnr
    
    # Rule 4: neither matches any true rekvnr → unknown
    return None

In [ ]:
df_merged["true_rekvnr"] = df_merged.apply(determine_true_rekvnr, axis=1)
print("Before dropping: ", df_merged.shape)

# Rows WITH a valid pathology rekvnr
df_valid = df_merged[df_merged["true_rekvnr"].notna()].copy()

# Rows WITHOUT a valid pathology rekvnr
df_invalid = df_merged[df_merged["true_rekvnr"].isna()].copy()

print("Rows dropped: ", len(df_invalid))
print("After dropping: ", len(df_valid))

In [ ]:
# Save to CSV for later use
output_file = r"D:\DATA\full_dataset\rekvnr_not_found.csv"
df_invalid.to_csv(output_file, index=False)

print(f"Saved {len(df_invalid)} records to {output_file}")

In [ ]:
print(df_valid.head(2))

In [ ]:
# Drop column with ocr rekvnr
df_valid = df_valid.drop(columns=["rekvnr", "file_size", "data_folder_size", "ocr_rekvnr"])
df_valid = df_valid.rename(columns={"true_rekvnr": "rekvnr"})
df_valid["rekvnr"] = pd.to_numeric(df_valid["rekvnr"], errors="coerce").astype("Int64")
print(df_valid.shape)
print(df_valid.head(2))

In [ ]:
# Merge
df_combined = df_valid.merge(df_pathology, on="rekvnr", how="left")
print(df_combined.shape)
print(df_combined.head(2))

In [ ]:
import re
# Observered HE misreads
HE_map = ["HE", "HF", "HE FR", "HE FRP", "4E", "HC", "AE", "HEE", "KE", "IE","HEI", "HET", "HEZ", "HF3", 'HE1', 'HE2', 'HE3', 'HE4']

# Valid observed stain
valid_stains = ['AFP', 'ALFAINHIBIN', 'ALK', 'ALK1', 'ALKNCL', 'ASMA', 'BCL2', 'BCL6', 'CA125', 'CAL', 'CD10', 'CD117', 'CD11C', 'CD138', 
                'CD14', 'CD15', 'CD1A', 'CD2', 'CD20', 'CD21', 'CD23', 'CD3', 'CD30', 'CD31', 'CD34', 'CD38', 'CD4', 'CD5', 'CD56', 'CD61', 
                'CD68KP1', 'CD68PGM1', 'CD7', 'CD79A', 'CD8', 'CD99', 'CDX2', 'CEA', 'CEAPO', 'CGA', 'CK19', 'CK20', 'CK5', 'CK7', 'CK8', 'CKAE1',
                'CKAE13', 'CMV', 'CMYC', 'COL4', 'CR', 'CYCD1', 'D240', 'DES', 'EBV', 'EMA', 'EP4', 'ER', 'FASCIN', 'FE', 'GCDFP', 'GFAP', 'GIEMSA',
                'GRAM', 'GROCOTT', 'HBME1', 'HCG', 'HEDYB', 'HEPA', 'HEV90', 'HMB45', 'HPL', 'HPV', 'IGK', 'IGL', 'IGM', 'KI67', 'KI67MLA', 
                'KONGO', 'LAM5', 'LCA', 'MAMGLOB', 'MCT', 'MGG', 'MLA', 'MLAR', 'MPO', 'MSB', 'MUC5AC', 'NAPSIN', 'ORCEIN', 'P16', 'P40', 'P53', 
                'P57', 'P63', 'PAONAPSIN', 'PAS', 'PASAB', 'PASD', 'PAX5', 'PIN', 'PIN1', 'PINDB', 'PLAP', 'PSA', 'RCC', 'RET', 'S100', 'SMAD4', 
                'SOX10', 'SPIRO', 'SYP', 'TB', 'TDT', 'TIA', 'TPO', 'TTF1', 'TTF1CK5', 'VG', 'VGAB', 'VIM', 'WADEFITE', 'WT1']

def clean_stain(s):
    if not s:
        return "unknown"

    # --- Normalize ---
    s = s.upper()
    s = re.sub(r'[^A-Z0-9 ]', '', s) # remove all characters that are not letter, digits or spaces
    s = re.sub(r'^\d+\s+', '', s)
    s = re.sub(r'(.)\1{2,}', r'\1', s) # collapse repeated characters
    s = s.strip() # remove leading/tailing whitespace

    if s in HE_map:
        return "HE"

    if s in valid_stains:
        return "other"

    return "unknown"

# Apply to dataframe
df_combined["stain"] = df_combined["ocr_stain"].astype(str).apply(clean_stain)

In [ ]:
print(df_combined.head(2))

In [ ]:
df_combined["stain"].value_counts()

In [ ]:
df_combined = df_combined.drop(columns=["ocr_stain"])

In [ ]:
# Save to csv
output_file = r"D:\DATA\full_dataset\overlapping_rekvnr.csv"
df_combined.to_csv(output_file, index=False)

print(f"Saved DataFrame to {output_file}")